In [0]:
gold_table = "workspace.ecommerce.user_predictions_gold"
spark.table(gold_table).printSchema()

In [0]:
import time
from pyspark.sql import functions as F

gold_table = "workspace.ecommerce.user_predictions_gold"

print("Running baseline heavy query...")

start_time = time.time()

baseline_df = (
    spark.table(gold_table)
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_records"),
        F.avg("purchase_probability").alias("avg_probability")
    )
)

baseline_df.display()

baseline_time = time.time() - start_time
print(f"Baseline Runtime: {baseline_time:.2f} seconds")

In [0]:
print("\nRunning optimized query...")

start_time = time.time()

optimized_df = (
    spark.table(gold_table)
    .select("user_id", "purchase_probability")   # Column pruning
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_records"),
        F.avg("purchase_probability").alias("avg_probability")
    )
)

optimized_df.display()

optimized_time = time.time() - start_time
print(f"Optimized Runtime: {optimized_time:.2f} seconds")

In [0]:
print("\nRunning filtered query (high probability users)...")

start_time = time.time()

filtered_df = (
    spark.table(gold_table)
    .select("user_id", "purchase_probability")
    .filter(F.col("purchase_probability") > 0.7)   # Pushdown filter
    .groupBy("user_id")
    .agg(F.count("*").alias("high_prob_count"))
)

filtered_df.display()

filtered_time = time.time() - start_time
print(f"Filtered Runtime: {filtered_time:.2f} seconds")

In [0]:
print("\n===== RUNTIME COMPARISON =====")
print(f"Baseline Runtime: {baseline_time:.2f} sec")
print(f"Optimized Runtime: {optimized_time:.2f} sec")
print(f"Filtered Runtime: {filtered_time:.2f} sec")

In [0]:
print("\nExplain Plan:")
optimized_df.explain(True)